# Cybersecurity Synthetic Dataset — Multiclass (Multinomial) Logistic Regression

This notebook focuses on **multiclass classification** of `attack_type` using **multinomial logistic regression** and **regularized variants**.

Models included:
- **Multinomial Logistic Regression (weak regularization ≈ unregularized)** — `C` very large
- **Multinomial Logistic Regression (L2 regularization)**
- **Multinomial Logistic Regression (L1 regularization)** via `LogisticRegressionCV` (solver=`saga`)
- **Multinomial Logistic Regression (Elastic Net)** via `LogisticRegressionCV` (solver=`saga`)
- **SGDClassifier (log-loss, elasticnet penalty)** as a fast scalable variant

Multi-color visualizations:
- Class distribution (bar plot)
- PCA projection (colored by class)
- Confusion matrix (heatmap)
- Per-class F1 scores (multi-color bar plot)
- Coefficient importance (top features per class) for the best regularized model

> **Note:** With high-dimensional, collinear features, **regularization typically helps**.


In [ ]:
# --- Imports and config ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.linear_model import SGDClassifier

from sklearn.metrics import (
    accuracy_score, f1_score, log_loss,
    confusion_matrix, classification_report
)

RANDOM_STATE = 7
np.random.seed(RANDOM_STATE)

# Dataset path (adjust if needed)
DATA_PATH = "cyber_15000.csv"

# Optional: speed knob (set to an int like 12000 to subsample rows; None = use all)
SUBSAMPLE_N = None  # e.g., 12000


In [ ]:
# --- Load dataset ---
df = pd.read_csv(DATA_PATH, low_memory=False)
df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], errors="coerce")

if SUBSAMPLE_N is not None and SUBSAMPLE_N < len(df):
    df = df.sample(SUBSAMPLE_N, random_state=RANDOM_STATE).reset_index(drop=True)

print("Shape:", df.shape)
display(df.head(3))


In [ ]:
# --- Quick sanity checks ---
print("Class distribution:")
display(df["attack_type"].value_counts())

print("\nTarget columns present?",
      {"attack_type" in df.columns, "incident_impact_score" in df.columns})


## 1) Exploratory visuals (multi-color)

- Attack-type distribution
- Impact score distribution (not used for classification but useful context)


In [ ]:
# --- Class distribution & impact distribution ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Class distribution (multi-color bars)
class_counts = df["attack_type"].value_counts()
classes = class_counts.index.tolist()
counts = class_counts.values

cmap = plt.cm.get_cmap("tab10", len(classes))
colors = [cmap(i) for i in range(len(classes))]

axes[0].bar(classes, counts, color=colors, edgecolor="white")
axes[0].set_title("Attack Type Distribution")
axes[0].set_xlabel("attack_type")
axes[0].set_ylabel("count")
axes[0].tick_params(axis="x", rotation=20)

# Impact distribution
axes[1].hist(df["incident_impact_score"].values, bins=40, edgecolor="white")
axes[1].set_title("Incident Impact Score Distribution")
axes[1].set_xlabel("incident_impact_score")
axes[1].set_ylabel("count")

plt.tight_layout()
plt.show()


## 2) Feature engineering

We add lightweight time features and create `X` / `y` for classification.

- `hour_utc`, `dayofweek_utc`
- boolean → 0/1
- drop `record_id` and raw `timestamp_utc`


In [ ]:
df_fe = df.copy()

# Add time features
df_fe["hour_utc"] = df_fe["timestamp_utc"].dt.hour.astype("Int64")
df_fe["dayofweek_utc"] = df_fe["timestamp_utc"].dt.dayofweek.astype("Int64")
for c in ["hour_utc", "dayofweek_utc"]:
    df_fe[c] = df_fe[c].fillna(int(df_fe[c].median()))

# bool -> int
for c in df_fe.select_dtypes(include=["bool"]).columns:
    df_fe[c] = df_fe[c].astype(np.int8)

# Targets
y = df_fe["attack_type"].astype(str)

# Drop targets + raw timestamp + ID
drop_cols = ["attack_type", "incident_impact_score", "timestamp_utc", "record_id"]
X = df_fe.drop(columns=[c for c in drop_cols if c in df_fe.columns])

print("X shape:", X.shape, "y shape:", y.shape)
display(X.head(3))


## 3) Train/test split (stratified)

We stratify so each split keeps similar proportions of attack types.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Train:", X_train.shape, "Test:", X_test.shape)


## 4) PCA visualization (colored by class)

A quick 2D PCA projection to see structure and overlap between classes.


In [ ]:
# Standardize for PCA + linear models
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

pca = PCA(n_components=2, random_state=RANDOM_STATE)
Z = pca.fit_transform(X_train_s)

classes = sorted(y_train.unique())
cmap = plt.cm.get_cmap("tab10", len(classes))
color_map = {c: cmap(i) for i, c in enumerate(classes)}

plt.figure(figsize=(10, 7))
for c in classes:
    idx = (y_train.values == c)
    plt.scatter(Z[idx, 0], Z[idx, 1], s=10, alpha=0.55, color=color_map[c], label=c)

plt.title("PCA (2D) of Standardized Features (Train Set)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend(markerscale=2, frameon=True)
plt.tight_layout()
plt.show()

print("Explained variance ratio:", pca.explained_variance_ratio_)


## 5) Models to compare

### A) Weak regularization (≈ unregularized)
`C` extremely large means very weak L2 shrinkage, which often overfits on noisy/collinear features.

### B) L2-regularized multinomial logistic regression
Strong regularization often improves generalization.

### C) L1-regularized multinomial logistic regression (saga)
Encourages sparsity (feature selection).

### D) Elastic Net multinomial logistic regression (saga)
Balances sparsity and stability.

### E) SGDClassifier (log-loss) with elastic net
Often faster on large/high-dimensional data.


In [ ]:
def evaluate_classifier(name, clf, X_train_s, X_test_s, y_train, y_test):
    clf.fit(X_train_s, y_train)
    pred = clf.predict(X_test_s)

    # Predict probabilities if available (for log loss)
    if hasattr(clf, "predict_proba"):
        proba = clf.predict_proba(X_test_s)
        ll = float(log_loss(y_test, proba, labels=sorted(y_train.unique())))
    else:
        proba = None
        ll = np.nan

    return {
        "model": name,
        "accuracy": float(accuracy_score(y_test, pred)),
        "macro_f1": float(f1_score(y_test, pred, average="macro")),
        "weighted_f1": float(f1_score(y_test, pred, average="weighted")),
        "log_loss": ll,
        "pred": pred,
        "proba": proba,
        "estimator": clf,
    }

# --- Model definitions ---
# Weak regularization (≈ unregularized)
clf_weak = LogisticRegression(
    multi_class="multinomial",
    solver="lbfgs",
    penalty="l2",
    C=1e6,           # very weak regularization
    max_iter=600,
)

# Stronger L2 regularization (good default)
clf_l2 = LogisticRegression(
    multi_class="multinomial",
    solver="lbfgs",
    penalty="l2",
    C=0.35,
    max_iter=600,
)

# L1 LogisticRegressionCV (saga)
Cs = np.logspace(-3, 2, 10)
clf_l1_cv = LogisticRegressionCV(
    Cs=Cs,
    cv=3,
    scoring="f1_macro",
    multi_class="multinomial",
    solver="saga",
    penalty="l1",
    max_iter=2000,
    n_jobs=-1,
    refit=True,
    random_state=RANDOM_STATE
)

# Elastic Net LogisticRegressionCV (saga)
clf_en_cv = LogisticRegressionCV(
    Cs=Cs,
    cv=3,
    scoring="f1_macro",
    multi_class="multinomial",
    solver="saga",
    penalty="elasticnet",
    l1_ratios=[0.15, 0.35, 0.5, 0.65, 0.85],
    max_iter=2500,
    n_jobs=-1,
    refit=True,
    random_state=RANDOM_STATE
)

# SGDClassifier (fast) with elastic net (log-loss)
clf_sgd = SGDClassifier(
    loss="log_loss",
    penalty="elasticnet",
    alpha=1e-4,
    l1_ratio=0.15,
    max_iter=5000,
    tol=1e-3,
    random_state=RANDOM_STATE
)

models = [
    ("LogReg weak reg (C=1e6, ≈ none)", clf_weak),
    ("LogReg L2 strong (C=0.35)", clf_l2),
    ("LogReg L1 (CV, saga)", clf_l1_cv),
    ("LogReg ElasticNet (CV, saga)", clf_en_cv),
    ("SGDClassifier (elasticnet)", clf_sgd),
]

results = []
for name, clf in models:
    results.append(evaluate_classifier(name, clf, X_train_s, X_test_s, y_train, y_test))

res_df = pd.DataFrame([{k: v for k, v in r.items() if k in ["model", "accuracy", "macro_f1", "weighted_f1", "log_loss"]} for r in results])
res_df = res_df.sort_values(by="macro_f1", ascending=False).reset_index(drop=True)

display(res_df)


## 6) Multi-color model comparison plot

We plot Accuracy and Macro-F1 with distinct colors per model.


In [ ]:
models_order = res_df["model"].tolist()
acc = res_df["accuracy"].values
mf1 = res_df["macro_f1"].values

cmap = plt.cm.get_cmap("tab20", len(models_order))
colors = [cmap(i) for i in range(len(models_order))]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].barh(models_order, acc, color=colors, edgecolor="white")
axes[0].set_title("Accuracy by Model")
axes[0].set_xlabel("Accuracy")

axes[1].barh(models_order, mf1, color=colors, edgecolor="white")
axes[1].set_title("Macro-F1 by Model (primary)")
axes[1].set_xlabel("Macro-F1")

plt.tight_layout()
plt.show()


## 7) Confusion matrix for the best model (multi-color heatmap)

We select the best model by **Macro-F1** and visualize the confusion matrix.


In [ ]:
best_name = res_df.loc[0, "model"]
best = next(r for r in results if r["model"] == best_name)

print("Best model:", best_name)
print("Accuracy:", best["accuracy"], "Macro-F1:", best["macro_f1"], "Weighted-F1:", best["weighted_f1"])

labels = sorted(y_train.unique())
cm = confusion_matrix(y_test, best["pred"], labels=labels)

plt.figure(figsize=(8.5, 7.5))
plt.imshow(cm, interpolation="nearest", cmap="viridis")
plt.title(f"Confusion Matrix — {best_name}")
plt.colorbar(label="count")
tick_marks = np.arange(len(labels))
plt.xticks(tick_marks, labels, rotation=45, ha="right")
plt.yticks(tick_marks, labels)

# Annotate counts
thresh = cm.max() * 0.55
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, str(cm[i, j]),
                 ha="center", va="center",
                 color="white" if cm[i, j] > thresh else "black",
                 fontsize=9)

plt.ylabel("True label")
plt.xlabel("Predicted label")
plt.tight_layout()
plt.show()


## 8) Per-class F1 (multi-color bar plot)

This helps identify which attack types benefit most from regularization.


In [ ]:
report = classification_report(y_test, best["pred"], labels=labels, output_dict=True, zero_division=0)
per_class_f1 = {k: v["f1-score"] for k, v in report.items() if k in labels}

plt.figure(figsize=(10, 5))
cmap = plt.cm.get_cmap("tab10", len(labels))
colors = [cmap(i) for i in range(len(labels))]

plt.bar(list(per_class_f1.keys()), list(per_class_f1.values()), color=colors, edgecolor="white")
plt.title(f"Per-class F1 — {best_name}")
plt.xlabel("class")
plt.ylabel("F1-score")
plt.ylim(0, 1.0)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

print("Classification report (summary):")
display(pd.DataFrame(report).T.loc[labels + ["accuracy", "macro avg", "weighted avg"]])


## 9) Coefficient importance (top features per class)

For multinomial logistic regression, each class has its own coefficient vector.

We plot the **top 15 absolute coefficients** for each class for the best model **if** it exposes `coef_`.
This works for LogisticRegression / LogisticRegressionCV (not for SGDClassifier unless using `coef_` too).


In [ ]:
feature_names = X.columns.to_list()
est = best["estimator"]

if not hasattr(est, "coef_"):
    print("Best estimator does not expose coef_. Try selecting a LogisticRegression-based model.")
else:
    coefs = est.coef_  # shape: (n_classes, n_features)
    classes = est.classes_

    for i, cls in enumerate(classes):
        w = coefs[i]
        top_k = 15
        top_idx = np.argsort(np.abs(w))[-top_k:][::-1]
        top_feats = [feature_names[j] for j in top_idx]
        top_vals = w[top_idx]

        # Color by sign
        colors = [plt.cm.RdBu(0.85) if v > 0 else plt.cm.RdBu(0.15) for v in top_vals]

        plt.figure(figsize=(10, 6))
        plt.barh(top_feats[::-1], top_vals[::-1], color=colors[::-1], edgecolor="white")
        plt.title(f"Top coefficients for class: {cls}")
        plt.xlabel("Coefficient value (signed)")
        plt.tight_layout()
        plt.show()

    # If CV-based, print chosen hyperparameters
    if hasattr(est, "C_"):
        print("Chosen C (per class):", est.C_)
    if hasattr(est, "l1_ratio_"):
        print("Chosen l1_ratio (per class):", est.l1_ratio_)


## 10) Takeaways

- Compare **weak regularization (≈ none)** vs **strong regularization**:
  - If Macro-F1 improves with stronger regularization, the dataset is benefiting from shrinkage.
- **L1** (sparse) and **Elastic Net** can reduce variance and ignore noisy/redundant features.
- PCA and the confusion matrix often explain *where* the model is confused (e.g., phishing vs malware).

### Practical suggestions
- If training time is long:
  - set `SUBSAMPLE_N = 12000` or `8000`
  - reduce `cv` from 3 to 2 in `LogisticRegressionCV`
  - reduce the `Cs` grid size
